# Demo 2: HKR Loss for Binary Classification


The HKR loss leverages Lipschitz constraints to enforce large margins. Below we minimise it on a toy dataset.


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from deel.lip.layers import SpectralDense, GroupSort2
from deel.lip.losses import HKR
from deel.lip.model import Sequential


In [ ]:
torch.manual_seed(5)
data_pos = torch.randn(256, 2) + torch.tensor([2.0, 0.0])
data_neg = torch.randn(256, 2) + torch.tensor([-2.0, 0.0])
inputs = torch.cat([data_pos, data_neg], dim=0)
labels = torch.cat([torch.ones(256), torch.zeros(256)], dim=0)
dataset = TensorDataset(inputs, labels.unsqueeze(-1))
loader = DataLoader(dataset, batch_size=64, shuffle=True)


In [ ]:
model = Sequential(
    SpectralDense(2, 32, activation="relu", use_bias=False),
    GroupSort2(),
    SpectralDense(32, 1, use_bias=False),
)
loss_fn = HKR(min_margin=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)


In [ ]:
def to_two_columns(y):
    return torch.cat([y, 1.0 - y], dim=-1)

for epoch in range(5):
    epoch_loss = 0.0
    for batch_inputs, batch_labels in loader:
        optimizer.zero_grad()
        logits = model(batch_inputs)
        logits = torch.cat([logits, -logits], dim=-1)
        loss = loss_fn(to_two_columns(batch_labels), logits)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_inputs.size(0)
    print(f"Epoch {epoch + 1}: loss={(epoch_loss / len(dataset)):.4f}")
